   # 01 – Data Review
   Goal: load all source tables, check their structure and quality, and list issues to fix in the cleaning stage.

In [1]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version();")).scalar())

PostgreSQL 18.6 on x86_64-windows, compiled by msvc-19.44.35229, 64-bit


In [2]:
import pandas as pd

RAW = "../data/raw/"

sales     = pd.read_csv(RAW + "Candy_Sales.csv", dtype={"Postal Code": str})
products  = pd.read_csv(RAW + "Candy_Products.csv")
factories = pd.read_csv(RAW + "Candy_Factories.csv")
targets   = pd.read_csv(RAW + "Candy_Targets.csv")
zips      = pd.read_csv(RAW + "uszips.csv", dtype={"zip": str})

In [3]:
dictionary = pd.read_csv(RAW + "candy_distributor_data_dictionary.csv").dropna(how="all")
display(dictionary)

,Table,Field,Description
0,Sales,Row ID,Unique row identifier
1,Sales,Order ID,Unique order identifier
2,Sales,Order Date,Date of order
3,Sales,Ship Date,Date of shipment
4,Sales,Ship Mode,Shipping method of order
5,Sales,Customer ID,Unique customer identifier
6,Sales,Country/Region,Country or region of customer
7,Sales,City,City of customer
8,Sales,State/Province,State/province of customer
9,Sales,Postal Code,Postal code / zip code of customer


In [4]:
for name, df in {"sales": sales, "products": products, "factories": factories,
                 "targets": targets, "zips": zips}.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

sales: 10194 rows, 18 columns
products: 15 rows, 6 columns
factories: 5 rows, 3 columns
targets: 3 rows, 2 columns
zips: 33787 rows, 18 columns


In [5]:
sales.info()


<class 'pandas.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          10194 non-null  int64  
 1   Order ID        10194 non-null  str    
 2   Order Date      10194 non-null  str    
 3   Ship Date       10194 non-null  str    
 4   Ship Mode       10194 non-null  str    
 5   Customer ID     10194 non-null  int64  
 6   Country/Region  10194 non-null  str    
 7   City            10194 non-null  str    
 8   State/Province  10194 non-null  str    
 9   Postal Code     10194 non-null  str    
 10  Division        10194 non-null  str    
 11  Region          10194 non-null  str    
 12  Product ID      10194 non-null  str    
 13  Product Name    10194 non-null  str    
 14  Sales           10194 non-null  float64
 15  Units           10194 non-null  int64  
 16  Gross Profit    10194 non-null  float64
 17  Cost            10194 non-null  float64
dt

**sales – findings:**
- 10,194 rows, 18 columns, no missing values
- Numeric columns have correct types
- `Order Date` and `Ship Date` are stored as text → convert to datetime in cleaning

In [6]:
for col in ["Ship Mode", "Country/Region", "Region", "Division"]:
    counts = sales[col].value_counts()
    share = (sales[col].value_counts(normalize=True) * 100).round(1)
    display(pd.DataFrame({"count": counts, "share_%": share}))

,count,share_%
Ship Mode,,
Standard Class,6120,60.0
Second Class,1979,19.4
First Class,1548,15.2
Same Day,547,5.4


,count,share_%
Country/Region,,
United States,9994,98.0
Canada,200,2.0


,count,share_%
Region,,
Pacific,3253,31.9
Atlantic,2986,29.3
Interior,2335,22.9
Gulf,1620,15.9


,count,share_%
Division,,
Chocolate,9844,96.6
Other,310,3.0
Sugar,40,0.4


**Categorical columns – findings:**
- No typos or unexpected categories
- 200 orders from Canada (2%) – postal codes to be checked against US ZIP table
- Chocolate dominates (96,6% of order lines); Sugar has only 40 lines → limited conclusions for this division

In [7]:
order_date = pd.to_datetime(sales["Order Date"])
ship_date = pd.to_datetime(sales["Ship Date"])

print("Order Date:", order_date.min(), "→", order_date.max())
print("Ship Date: ", ship_date.min(), "→", ship_date.max())
print((ship_date - order_date).dt.days.describe())

Order Date: 2021-01-03 00:00:00 → 2024-12-30 00:00:00
Ship Date:  2026-06-30 00:00:00 → 2030-06-28 00:00:00
count    10194.000000
mean      2003.961840
std          1.744106
min       2000.000000
25%       2003.000000
50%       2004.000000
75%       2005.000000
max       2011.000000
dtype: float64


In [8]:
lead_time = (ship_date - order_date).dt.days - 2000
display(lead_time.groupby(sales["Ship Mode"]).describe())

,count,mean,std,min,25%,50%,75%,max
Ship Mode,,,,,,,,
First Class,1548.0,2.182171,0.771431,1.0,2.0,2.0,3.0,4.0
Same Day,547.0,0.043876,0.205006,0.0,0.0,0.0,0.0,1.0
Second Class,1979.0,3.237494,1.191556,1.0,2.0,3.0,4.0,5.0
Standard Class,6120.0,4.996405,1.019977,3.0,4.0,5.0,6.0,11.0


**Dates – findings:**
- Orders: 2021-01-03 → 2024-12-30; ship dates: 2026-06-30 → 2030-06-28 (impossible)
- Order-to-ship gap is 2000–2011 days → ship dates appear shifted by a constant 2000 days
- Hypothesis check: after subtracting 2000 days, lead time follows ship mode logically
  (Same Day ≈ 0 days, First Class ≈ 2, Second Class ≈ 3, Standard Class ≈ 5)
- Decision for cleaning: subtract 2000 days from `Ship Date`

In [9]:
print("Duplicate rows:", sales.duplicated().sum())
print("Duplicate Row IDs:", sales["Row ID"].duplicated().sum())

diff = (sales["Sales"] - sales["Gross Profit"] - sales["Cost"]).abs()
print("Rows where Sales ≠ Gross Profit + Cost:", (diff > 0.01).sum())

Duplicate rows: 0
Duplicate Row IDs: 0
Rows where Sales ≠ Gross Profit + Cost: 0


**Integrity – findings:**
- No duplicate rows, no duplicate Row IDs
- Sales = Gross Profit + Cost in every row → financial data is consistent

In [10]:
display(products)
display(factories)
display(targets)

,Division,Product Name,Factory,Product ID,Unit Price,Unit Cost
0,Chocolate,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,CHO-NUT-13000,3.49,1.00
1,Chocolate,Wonka Bar - Fudge Mallows,Lot's O' Nuts,CHO-FUD-51000,3.60,1.20
2,Chocolate,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,CHO-SCR-58000,3.60,1.10
3,Chocolate,Wonka Bar - Milk Chocolate,Wicked Choccy's,CHO-MIL-31000,3.25,1.14
4,Chocolate,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,CHO-TRI-54000,3.75,1.30
5,Sugar,Laffy Taffy,Sugar Shack,SUG-LAF-25000,1.99,0.75
6,Sugar,SweeTARTS,Sugar Shack,SUG-SWE-91000,1.50,0.80
7,Sugar,Nerds,Sugar Shack,SUG-NER-92000,1.50,0.80
8,Sugar,Fun Dip,Sugar Shack,SUG-FUN-75000,1.50,0.90
9,Other,Fizzy Lifting Drinks,Sugar Shack,OTH-FIZ-56000,3.75,1.50


,Factory,Latitude,Longitude
0,Lot's O' Nuts,32.881893,-111.768036
1,Wicked Choccy's,32.076176,-81.088371
2,Sugar Shack,48.119140,-96.181150
3,Secret Factory,41.446333,-90.565487
4,The Other Factory,35.117500,-89.971107


,Division,Target
0,Chocolate,27000
1,Sugar,15000
2,Other,3000


**Products, factories, targets – findings:**
- 15 products; each product is made in exactly one factory, while a factory can make several products (one-to-many)
- Chocolate is produced in two factories (Lot's O' Nuts – Arizona, Wicked Choccy's – Georgia) → orders could realistically be reassigned between them
- Factory coordinates look valid (all within the US)
- Kazookles has a very thin margin (price 3.25, cost 3.00)
- Targets have no year or unit → to be verified against actual sales

In [11]:
yearly_sales = (
    sales.assign(Year=order_date.dt.year)
    .pivot_table(index="Division", columns="Year", values="Sales", aggfunc="sum")
    .round(0)
)
display(yearly_sales.join(targets.set_index("Division")))

,2021,2022,2023,2024,Target
Division,,,,,
Chocolate,26356.0,27763.0,34193.0,43380.0,27000
Other,2549.0,1208.0,2462.0,3444.0,3000
Sugar,48.0,32.0,204.0,144.0,15000


**Targets – findings:**
- Data dictionary: `Target` = sales target for 2024 only
- 2024 achievement: Chocolate ~161%, Other ~115%, Sugar ~1% (144 vs 15,000) → Sugar target inconsistent with actual sales
- Decision: targets are not needed to answer the business questions (transportation costs) → excluded from the analysis

In [12]:
us = sales["Country/Region"] == "United States"
postal = sales["Postal Code"]

print("Postal code lengths:\n", postal.str.len().value_counts(), "\n")
print("US codes not found in zips (as is):", (~postal[us].isin(zips["zip"])).sum())
print("US codes not found in zips (padded to 5 digits):",
      (~postal[us].str.zfill(5).isin(zips["zip"].str.zfill(5))).sum())

display(sales.loc[~us, ["City", "State/Province", "Postal Code"]].head())

Postal code lengths:
 Postal Code
5    9994
3     200
Name: count, dtype: int64 

US codes not found in zips (as is): 0
US codes not found in zips (padded to 5 digits): 0


,City,State/Province,Postal Code
113,Vancouver,British Columbia,V5K
127,Vancouver,British Columbia,V5K
225,Toronto,Ontario,M2N
227,Toronto,Ontario,M2N
238,Toronto,Ontario,M2N


**Postal codes – findings:**
- All 9,994 US postal codes have 5 digits and match the US ZIP table (leading zeros preserved by reading as text)
- 200 Canadian orders have 3-character postal codes (e.g. V5K, M2N) not covered by the US ZIP table
- Decision: Canadian orders excluded from distance analysis (in line with Project Charter scope: US orders only)


In [13]:
print("Product IDs in sales missing from products:",
      (~sales["Product ID"].isin(products["Product ID"])).sum())
print("Factories in products missing from factories:",
      (~products["Factory"].isin(factories["Factory"])).sum())

Product IDs in sales missing from products: 0
Factories in products missing from factories: 0


**Table relationships – findings:**
- Every Product ID in sales exists in products; every factory in products has coordinates
- Chain order → product → factory → coordinates is complete for all orders

## Summary

**Data quality is good overall:** no missing values, no duplicates, financial data consistent, all tables join correctly.

**Issues and decisions for the cleaning stage:**

| Issue | Decision |
| --- | --- |
| `Order Date` and `Ship Date` stored as text | Convert to datetime |
| `Ship Date` shifted by a constant 2000 days (confirmed by lead times per ship mode) | Subtract 2000 days |
| 200 Canadian orders without matching coordinates | Exclude from distance analysis (scope: US only) |
| Sugar target inconsistent; targets not needed for business questions | Exclude targets table from analysis |
| Sugar division has only 40 order lines | Note as a limitation in conclusions |